# 02 — Cleaning + outcome resolution

This is where censoring gets handled properly — the part prior-art models skip.

Reads: `objects_slim.csv`, `funding_rounds.csv`, `acquisitions.csv`, `ipos.csv`
Writes: `data/processed/outcomes.csv` (one row per company, with a `duration_months` and `event` flag ready for survival modeling)

In [6]:
import pandas as pd
import numpy as np

RAW = "../data/raw"
PROCESSED = "../data/processed"

SNAPSHOT_DATE = pd.Timestamp("2013-12-12")  # dataset snapshot date — cite this in your report
STALE_MONTHS = 36  # presumed-dead threshold. Rerun this notebook at 24 and 48 too and report
                    # how much the censoring % and later the C-index shift — expect this question.

## Load and do basic cleaning

In [2]:
fr = pd.read_csv(f"{RAW}/funding_rounds.csv", encoding="ISO-8859-1", low_memory=False)
obj = pd.read_csv(f"{RAW}/objects_slim.csv", encoding="ISO-8859-1", low_memory=False)
acq = pd.read_csv(f"{RAW}/acquisitions.csv", encoding="ISO-8859-1", low_memory=False)
ipo = pd.read_csv(f"{RAW}/ipos.csv", encoding="ISO-8859-1", low_memory=False)

fr["funded_at"] = pd.to_datetime(fr["funded_at"], errors="coerce")
n0 = len(fr)
fr = fr.dropna(subset=["funded_at"])
fr = fr[fr["raised_amount_usd"].fillna(0) > 0]  # drop undisclosed rounds (=0/NaN) — NOT zero capital
print(f"[clean] funding_rounds: {n0} -> {len(fr)} after dropping undisclosed/undated rounds")

for col in ["founded_at", "closed_at", "last_funding_at", "first_funding_at"]:
    obj[col] = pd.to_datetime(obj[col], errors="coerce")
acq["acquired_at"] = pd.to_datetime(acq["acquired_at"], errors="coerce")
ipo["public_at"] = pd.to_datetime(ipo["public_at"], errors="coerce")

print(f"[clean] {len(obj):,} companies, {len(acq):,} acquisitions, {len(ipo):,} ipos")

[clean] funding_rounds: 52928 -> 46817 after dropping undisclosed/undated rounds
[clean] 196,553 companies, 9,562 acquisitions, 1,259 ipos


## Resolve outcomes

Four cases, in priority order:
1. **Closed** → event observed at `closed_at`
2. **Acquired / IPO'd** → censored at exit date (can't fail after being acquired)
3. **Operating but stale** (no funding activity for `STALE_MONTHS`+) → presumed dead
4. **Everything else** → genuinely right-censored at the snapshot date

The join below is LEFT, never INNER — an inner join here silently intersects
the cohort down to "companies that were both acquired AND went public,"
which is a real bug this exact pipeline hit once already.

In [3]:
df = obj.copy()

df = df.merge(
    acq[["acquired_object_id", "acquired_at"]].rename(columns={"acquired_object_id": "id"}),
    on="id", how="left",
)
df = df.merge(
    ipo[["object_id", "public_at"]].rename(columns={"object_id": "id"}),
    on="id", how="left",
)

df["event"] = 0
df["event_date"] = pd.NaT

closed = df["status"] == "closed"
df.loc[closed, "event"] = 1
df.loc[closed, "event_date"] = df.loc[closed, "closed_at"]

exited = (~closed) & (df["acquired_at"].notna() | df["public_at"].notna())
exit_date = df["acquired_at"].combine_first(df["public_at"])
df.loc[exited, "event"] = 0
df.loc[exited, "event_date"] = exit_date[exited]

still_open = ~closed & ~exited
last_seen = df["last_funding_at"].combine_first(df["first_funding_at"])
months_stale = (SNAPSHOT_DATE - last_seen).dt.days / 30.44
stale = still_open & (months_stale >= STALE_MONTHS)
df.loc[stale, "event"] = 1
df.loc[stale, "event_date"] = last_seen[stale] + pd.DateOffset(months=STALE_MONTHS)

remaining = still_open & ~stale
df.loc[remaining, "event"] = 0
df.loc[remaining, "event_date"] = SNAPSHOT_DATE

start = df["founded_at"].combine_first(df["first_funding_at"])
df["duration_months"] = (df["event_date"] - start).dt.days / 30.44
n_before = len(df)
df = df[df["duration_months"] > 0]
print(f"[outcomes] dropped {n_before - len(df)} rows with missing/inverted dates")

censor_pct = 100 * (1 - df["event"].mean())
print(f"[outcomes] N={len(df)}  events={int(df['event'].sum())}  censored={censor_pct:.1f}%")
print(f"[outcomes] stale-presumed-dead at STALE_MONTHS={STALE_MONTHS}: {int(stale.sum())} companies")
print()
print(f"^ THIS censored % IS YOUR HEADLINE NUMBER for the eval slide and IDF Experiment 2:")
print(f"  it's the fraction of data a fixed-horizon classifier deletes, that this pipeline retains.")

[outcomes] dropped 98424 rows with missing/inverted dates
[outcomes] N=98280  events=9428  censored=90.4%
[outcomes] stale-presumed-dead at STALE_MONTHS=36: 7054 companies

^ THIS censored % IS YOUR HEADLINE NUMBER for the eval slide and IDF Experiment 2:
  it's the fraction of data a fixed-horizon classifier deletes, that this pipeline retains.


## Sensitivity check on STALE_MONTHS — a panel/examiner will ask for this

In [4]:
def censor_rate_at(stale_months):
    ms = (SNAPSHOT_DATE - last_seen).dt.days / 30.44
    s = still_open & (ms >= stale_months)
    ev = closed.astype(int).copy()
    ev.loc[s] = 1
    return 100 * (1 - ev.mean()), int(s.sum())

for m in [24, 36, 48]:
    pct, n_stale = censor_rate_at(m)
    print(f"STALE_MONTHS={m}: censored={pct:.1f}%, presumed-dead-by-staleness={n_stale}")

STALE_MONTHS=24: censored=93.0%, presumed-dead-by-staleness=11158
STALE_MONTHS=36: censored=95.1%, presumed-dead-by-staleness=7054
STALE_MONTHS=48: censored=96.3%, presumed-dead-by-staleness=4738


## Save for the survival model notebook

In [5]:
import os
os.makedirs(PROCESSED, exist_ok=True)

keep_cols = ["id", "name", "category_code", "country_code", "status",
             "funding_total_usd", "funding_rounds", "duration_months", "event"]
df[keep_cols].to_csv(f"{PROCESSED}/outcomes.csv", index=False)
print(f"[done] wrote {PROCESSED}/outcomes.csv  ({len(df)} rows)")

[done] wrote ../data/processed/outcomes.csv  (98280 rows)
